# McFACTS Restructure Tutorial

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

import mcfacts

OUT = Path("./tutorial_runs")
OUT.mkdir(exist_ok=True)

## Command Line Interface

When McFACTS is installed, a command now gets added to your env.

Subcommands:

* `run`: run the simulation
* `plot`: use existing run data to create plots
* `rp` run the simulation and create plots

`-o=True` is short for `--overwrite_files=True` which will first delete the output folder if it exists.

In [ ]:
!mcfacts rp -o=True --galaxy_num=10 --output_dir=./tutorial_runs/cli

In [ ]:
run_dir = OUT / "cli"

for d in sorted(run_dir.iterdir()):
    print(d.name + ("/" if d.is_dir() else ""))

print(f"\nPlots: {sorted(x.name for x in (run_dir / "plots").iterdir())}")

display(Image(filename=f"{run_dir / "plots" / "gw_strain.png"}"))

## Lets change some settings!

Flags on the command line can change anything that isn't a protected/static setting.

The full list of default settings are in `src/mcfacts/inputs/settings_manager.py` and in `recipes/settings.ini` with comments

In [ ]:
!mcfacts run -o=True --galaxy_num=10 --active_timestep_num=100 --torque_prescription=jimenez_masset --output_dir=./tutorial_runs/cli_sm

In [ ]:
print("Unmodified:")
!grep -E "active_timestep_num|torque_prescription|galaxy_num" tutorial_runs/cli/settings.ini

print("\nModified:")
!grep -E "active_timestep_num|torque_prescription|galaxy_num" tutorial_runs/cli_sm/settings.ini

## Using a settings file

The .ini file is now sectioned into different categories. The order of precedence for settings changes goes as `command line flag > settings file > defaults`

In [ ]:
%%writefile tutorial_runs/new_settings.ini
[sim]
smbh_mass = 1e7
galaxy_num = 10

[disk]
torque_prescription = jimenez_masset

[io]
output_dir = ./tutorial_runs/cli_sm2
overwrite_files = True

In [ ]:
# Setting galaxy_num on the command line beats the 10 set in new_settings.ini

!mcfacts run -s=tutorial_runs/new_settings.ini --galaxy_num=5

print("\nSettings:")
!grep -E "smbh_mass|galaxy_num|torque_prescription|flag_phenom_turb" tutorial_runs/cli_sm2/settings.ini

## We now have a python-first approach

In [ ]:
from mcfacts.inputs.settings_manager import SettingsManager
from mcfacts import simulation

settings = SettingsManager({
    "galaxy_num": 5,
    "smbh_mass": 1e7,
    "torque_prescription": "jimenez_masset",
    "flag_phenom_turb": True,
    "output_dir": "./tutorial_runs/py_first",
    "overwrite_files": True,
})

print("Flat Access:", settings.smbh_mass)          # You can grab the value through an accessor
print("Through Category:", settings.sim.smbh_mass) # The category can be included to keep things neat :)

In [ ]:
simulation.main(settings)

## Building a simulation through python

`simulation.main` loops over the set of galaxies that compose our simulation. For each one it:
1. Creates a new `Galaxy` with its own seed and `FilingCabinet`
2. "Populates" the galaxy using `Populators`
3. Runs a one-step "Reclassification" to sort our initial input of objects
4. Runs the "Active Timeline" of ~36 actors for a number of timesteps set by `active_timestep_num`

We can do all of this by hand, in a separate script or in a notebook, to allow for the simulation to be customized every step of the way.

### Create the settings, disk, and an empty galaxy

In [ ]:
from mcfacts.objects.disk import AGNDisk
from mcfacts.objects.galaxy import Galaxy

# Create our settings manager with the simulation knobs we want to turn
settings = SettingsManager(
    {
        "active_timestep_num": 50,
        "output_dir": "./tutorial_runs/py_building"
    }
)

# Create an instance of the "AGN Disk" giving access to disk functions for the defined settings
agn_disk = AGNDisk(settings)

# Create a galaxy with a seed, and an id.
# The output dir needs to be defined here as well.
galaxy = Galaxy(seed=12345, galaxy_id='0', runs_folder=settings.output_dir, settings=settings)

print("Filing cabinet starts off empty:", list(galaxy.filing_cabinet.agn_objects))

### Populate!

A `Populator` builds a single population of objects and "files" it under a specified name.

We can use the `random_generator` created by the galaxy object to make sure results are reproducible.
Never call `np.random` directly!!!

In [ ]:
# Running this cell more than once will give you an error!
# This is the galaxy protecting its population and letting you know that it already has objects.

from mcfacts.objects.populators import SingleBlackHolePopulator

galaxy.populate([SingleBlackHolePopulator()], agn_disk)

print("Filing Cabinet:", list(galaxy.filing_cabinet.agn_objects))

In [ ]:
# Lets see what 'blackgoles_unsort' contains

def read_cabinet(galaxy):
    print(f"{'Array Name':<26} {'Array Type':<26} Array Length")

    for name, array in galaxy.filing_cabinet.agn_objects.items():
        print(f"{name:<26} {type(array).__name__:<26} {len(array):<5}")

read_cabinet(galaxy)

### Lets create a timeline

A `SimulationTimeline` is a list of actors that "act" on the population of a galaxy.

To create one, we need to give it that list, the number of timesteps it will run for, and the length of those timesteps in years.

This first one will be a "utility" timeline which will help us classify the objects we've populated.

In [ ]:
from mcfacts.objects.timeline import SimulationTimeline
from mcfacts.objects.actors import InitialBlackHoleReclassification, SingleBlackHoleRealityCheck

# Create a new timeline, setting a name, number of timesteps, and timestep length
# Here we only need to do one step of reclassification, and the timestep length doesn't matter.
reclass = SimulationTimeline("Reclassification", timesteps=1, timestep_length=0)

# Once we have a timeline created, we can add our actors
reclass.add_timeline_actors([
    InitialBlackHoleReclassification(), # Reclassify our objects in "blackholes_unsort" into prograde and retrograde orbiters
    SingleBlackHoleRealityCheck()       # Make sure all of our black holes are "real" and remove them if not.
])

# Run the timeline against the galaxy.
# Timelines can be used multiple times, and actors typically don't store any per-run information.
galaxy.run(reclass, agn_disk)

read_cabinet(galaxy)

### Lets take a closer look at the `AGNObjectArray`

We now have three arrays, `blackholes_unsort`, `blackholes_prograde`, and `blackholes_retrograde`.

Each array has a type that defines what type of object it holds. All of these arrays currently hold single black holes.

An `AGNObjectArray` is a set of parallel NumPy arrays, each row representing one object.

Every row has a `unique_id` (UUID), so objects should be referenced by id instead if index.

You can access an array through the `get_array` method of the filing cabinet. This is a reference, so any edits will be made directly to the objects.

In [ ]:
from mcfacts.objects.agn_object_array import AGNBlackHoleArray

bh_array_name = settings.bh_prograde_array_name # Each of our standard arrays are defined in settings, use this instead of typing out the string
print("settings.bh_prograde_array_name:", bh_array_name)

# We need to specify the type of object we expect from the filing cabinet.
# In this case, it is "AGNBlackHoleArray"
prograde = galaxy.filing_cabinet.get_array(bh_array_name, AGNBlackHoleArray)

print("\nAttributes:", list(prograde.get_super_dict())) # The "super dict" is a dictionary containing everything in the AGNObjectArray

print("First ids :", prograde.unique_id[:2])            # Attributes of the object can be accessed directly
print("Masses    :", prograde.mass[:3])

# We can get a list of unique ids using a mask
heavy_ids = prograde.unique_id[prograde.mass > 30]

# That list of ids can be used to grab all of the matching rows for a specific column
heavy_orb_a = prograde.get_attribute("orb_a", heavy_ids)

print("\norb_a of BHs > 30 Msun:", heavy_orb_a[:5])

In [ ]:
from mcfacts.modules.migration import ProgradeBlackHoleMigration
from mcfacts.modules.damping import ProgradeBlackHoleDamping
from mcfacts.modules.accretion import ProgradeBlackHoleAccretion

def create_new_galaxy(settings, seed=12345):
    g = Galaxy(seed=seed, runs_folder=settings.output_dir, galaxy_id="0", settings=settings)
    g.populate([SingleBlackHolePopulator()], agn_disk)

    t = SimulationTimeline("Reclassification", timesteps=1, timestep_length=0)
    t.add_timeline_actors([InitialBlackHoleReclassification(), SingleBlackHoleRealityCheck()])

    g.run(t, agn_disk)

    return g

def galaxy_summary(galaxy, label):
    pro = galaxy.filing_cabinet.get_array(bh_array_name, AGNBlackHoleArray)
    circ = np.sum(pro.orb_ecc <= settings.disk_bh_pro_orb_ecc_crit)

    # Compare each BH with its starting radius, matched by unique_id
    moved = sum(abs(a - START_ORB_A[uuid]) > 1 for uuid, a in zip(pro.unique_id, pro.orb_a))

    print(f"{label:24s} prograde:{len(pro):4d}  circularized:{circ:4d}  "f"migrated:{moved:4d}  total mass:{pro.mass.sum():7.1f} Msun")

def new_timeline(name):
    return SimulationTimeline(name, timesteps=settings.active_timestep_num, timestep_length=settings.active_timestep_duration_yr)

start = create_new_galaxy(settings)

start_pro = start.filing_cabinet.get_array(bh_array_name, AGNBlackHoleArray)
START_ORB_A = dict(zip(start_pro.unique_id, start_pro.orb_a))

galaxy_summary(start, "Initial Galaxy")

#### Add Migration

In [ ]:
galaxy_1 = create_new_galaxy(settings)

timeline_1 = new_timeline("Migration")

# This actor can act on different population groups, so we need to tell it which array we want to target
# In the canonical simulation we use this to run the same actors on our prograde and inner-disk population
timeline_1.add_timeline_actor(ProgradeBlackHoleMigration(target_array=bh_array_name))

galaxy_1.run(timeline_1, agn_disk)

galaxy_summary(galaxy_1, "Inital + migration")

Looks like nothing changed! Migration only acts on circularized objects, nothing currently starts out circularized, so lets add an actor to help with that.

#### Add Eccentricity Damping

In [ ]:
galaxy_2 = create_new_galaxy(settings)

timeline_2 = new_timeline("Migration + ecc damping")
timeline_2.add_timeline_actors([
    ProgradeBlackHoleMigration(target_array=bh_array_name),
    ProgradeBlackHoleDamping(target_array=bh_array_name)
])

galaxy_2.run(timeline_2, agn_disk)

galaxy_summary(galaxy_2, "Migration + ecc damping")

Sweet! Looks like we've got some circularization and migration happening. Lets add in some accretion and a reality check.

#### Add Accretion and Reality Check

In [ ]:
galaxy_3 = create_new_galaxy(settings)

timeline_3 = new_timeline("Migration + damping + accretion")
timeline_3.add_timeline_actors([
    ProgradeBlackHoleMigration(target_array=bh_array_name),
    ProgradeBlackHoleDamping(target_array=bh_array_name),
    ProgradeBlackHoleAccretion(target_array=bh_array_name),
    SingleBlackHoleRealityCheck()
])

# The default print statement for a timeline will list all of the actors it has in run-order
print(timeline_3)

galaxy_3.run(timeline_3, agn_disk)

galaxy_summary(galaxy_3, "Migration + damping + accretion")

# We can also look at the history of timelines run on a galaxy.
# Our "create_new_galaxy" method runs the reclassification timeline.
print("\nTimelines run:", [t.name for t in galaxy_3.timeline_history])

#### The Canonical Simulation

`simulation.run_galaxy()` builds the canonical timelines used by the `mcfacts run` command.

`run_galaxy()` in `src/mcfacts/simulation.py` is about 150 lines long, it also doubles as a "table of contents" for the physics we're running.

In [ ]:
galaxy_canon = Galaxy(seed=12345, runs_folder=settings.output_dir, galaxy_id="0", settings=settings)

galaxy_canon.populate([SingleBlackHolePopulator()], agn_disk)

simulation.run_galaxy(settings, galaxy_canon, agn_disk)

read_cabinet(galaxy_canon)

In [ ]:
# Lets print out the active timeline and see what actors we ran
print(galaxy_canon.timeline_history[1])

### Writing our own timeline actors

Physics and other operations go inside a `TimelineActor`.

Each actor takes a name, a constructor, and `perform(...)`, which edits the cabinet in-place and returns nothing.

Every actor gets the same arguments every timestep. If you didn't pass settings, the actor defaults to the settings of the galaxy.

**Goal:** *Lets create an actor that saves a value every timestep. That value can be used by other modules, or be used to track things in post-processing.*

In [ ]:
from mcfacts.objects.timeline import TimelineActor

# If this is your first introduction to object-orientated programing, you've probably never heard of a 'class' before.
# A class can be thought of as a container. The container can be copied, filled and emptied with whatever you want, and take any shape.
# In 'strickly-typed' languages (like Java) each "class" typically exists in its own file. Python allows for a more modular approach.

class TrackCircularized(TimelineActor):
    # When we create a new instance of "TrackCircularized" this method gets called.
    # For example: new_actor = TrackCircularized()
    # The "self" attribute exists only in the class and allows the method to reference stuff in the "container"
    def __init__(self, target_array, name=None, settings=None):
        # Take a look at the 'class' line above. It has the name of this class "TrackCircularized" and then "TimelineActor" in parentheses.
        # The name in the parentheses defines the "parent" class. This provides a common vocabulary for anything interacting with an actor.
        super().__init__("Track Circularized" if name is None else name, settings)

        self.target_array = target_array

    # This is the "meat" of the class. This is the method that the galaxy calls to run physics or any other operations.
    def perform(self, timestep, timestep_length, time_passed, filing_cabinet, agn_disk, random_generator):

        # If your actor requires an array, you need to check if it exists first!
        # Arrays come and go, and it may be possible that a required array doesn't exist when this actor is called.
        if self.target_array not in filing_cabinet:
            return

        # The `filing_cabinet` was passed into the method. We can get a reference to the array we care about from it.
        bh = filing_cabinet.get_array(self.target_array, AGNBlackHoleArray)

        # Count up the number of prograde bh that are at or below the critical eccentricity (circularized)
        n_circ = int(np.sum(bh.orb_ecc <= self.settings.disk_bh_pro_orb_ecc_crit))

        # The filing cabinet can hold arrays and values.
        # We read from the prograde array above. To track the number of circularized, lets create a value.
        if "n_circularized" in filing_cabinet:
            filing_cabinet.get_value("n_circularized").append(n_circ)
        else:
            filing_cabinet.set_value("n_circularized", [n_circ])

In [ ]:
runs = {
    "Migration only": [ProgradeBlackHoleMigration(target_array=bh_array_name)],
    "Migration + damping": [ProgradeBlackHoleMigration(target_array=bh_array_name), ProgradeBlackHoleDamping(target_array=bh_array_name)],
}

histories = {}

for label, actors in runs.items():
    gal = create_new_galaxy(settings)

    tl = new_timeline(label)
    tl.add_timeline_actors(actors + [TrackCircularized(target_array=bh_array_name)])

    gal.run(tl, agn_disk)

    histories[label] = gal.filing_cabinet.get_value("n_circularized")

time_myr = np.arange(settings.active_timestep_num) * settings.active_timestep_duration_yr / 1e6

fig, ax = plt.subplots(figsize=(5.5, 3.2))

for (label, hist), color in zip(histories.items(), ["#8a8f98", "#2a6fdb"]):
    ax.plot(time_myr, hist, color=color, lw=2, label=label)
    ax.annotate(label, (time_myr[-1], hist[-1]), xytext=(4, 0), textcoords="offset points",
                va="center", fontsize=9, color="0.25")

ax.set_xlabel("Time (Myr)")
ax.set_ylabel("Circularized prograde BHs")

ax.set_xlim(0, time_myr[-1] * 1.45)
ax.set_xticks(np.arange(0, time_myr[-1] + 1e-9, 0.1))

ax.grid(color="0.9", lw=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")

plt.tight_layout()
plt.show()

### Moving objects between arrays

Most of the bookkeeping we do is moving objects from one array to another.

The pattern to follow is:
1. **copy** the source array,
2. **keep only** the ids you want in the copy,
3. **remove** those ids from the source,
4. **append** the copy to the filing cabinet.

Lets create an actor that moves prograde BHs near the migration trap into their own array.

The definition of "near" is a **custom setting** created for this actor. Third-party settings go in their own category, reached as `settings.<category>.<name>`, and are saved to `settings.ini` under their own section.

In [ ]:
class CaptureAtTrap(TimelineActor):
    def __init__(self, name=None, settings=None):
        super().__init__("Capture At Trap" if name is None else name, settings)

    def perform(self, timestep, timestep_length, time_passed, filing_cabinet, agn_disk, random_generator):
        # We can create a quick reference for setting so we don't have to type self.settings.x.y every time.
        sm = self.settings

        if sm.bh_prograde_array_name not in filing_cabinet:
            return

        pro = filing_cabinet.get_array(sm.bh_prograde_array_name, AGNBlackHoleArray)

        near = np.abs(pro.orb_a - sm.disk_radius_trap) < sm.toy.trap_window * sm.disk_radius_trap
        ids = pro.unique_id[near]

        if len(ids) == 0:
            return

        moved = pro.copy()                                                   # 1 copy
        moved.keep_only(ids)                                                 # 2 keep only in copy
        pro.remove_all(ids)                                                  # 3 remove from source
        filing_cabinet.create_or_append_array("blackholes_at_trap", moved)   # 4 append copy to cabinet

        self.log(f"Moved {len(ids)} BHs to blackholes_at_trap") # To print from within an actor, use `self.log`

In [ ]:
settings = SettingsManager({"active_timestep_num": 50, "output_dir": "./tutorial_runs/by_hand"})

# Creating our custom category with a custom setting that isn't in default.
settings.add_custom_category("toy", {"trap_window": 0.5})

galaxy = create_new_galaxy(settings)

active = new_timeline("Migration + toys")
active.add_timeline_actors([
    ProgradeBlackHoleMigration(target_array=bh_array_name),     # order 0
    ProgradeBlackHoleAccretion(target_array=bh_array_name),     # 1000
    ProgradeBlackHoleDamping(target_array=bh_array_name),       # 2000
    SingleBlackHoleRealityCheck(),                              # 3000
])

active.add_timeline_actor(TrackCircularized(target_array=bh_array_name))  # 4000

active.add_timeline_actor(CaptureAtTrap(), relative_order=500)    # Will be added between migration and accretion

print(active)

galaxy.run(active, agn_disk)

read_cabinet(galaxy)

print("\nn_circularized each timestep:", galaxy.filing_cabinet.get_value("n_circularized"))